# 02 — Exploratory Data Analysis: Food Delivery Time

This notebook explores the food delivery time prediction dataset.
All core logic lives in `src/dabba/` — this notebook calls those functions.

**Dataset:** Food Delivery Time (Kaggle — rajatkumar30/food-delivery-time)
**Goal:** Understand delivery patterns, time distributions, and feature relationships.

In [ ]:
import sys
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from dabba.config import get_config
from dabba.data.loaders import load_delivery, describe_dataset
from dabba.data.cleaning import clean_delivery

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
config = get_config()

## 1. Load Raw Data

In [ ]:
df_raw = load_delivery(config)
describe_dataset(df_raw, 'Delivery (raw)')

## 2. Data Quality Check

In [ ]:
print('Shape:', df_raw.shape)
print('\nColumn types:')
print(df_raw.dtypes)
print('\nNull counts:')
print(df_raw.isnull().sum())

## 3. Target Variable: Delivery Time

In [ ]:
target_col = 'Time_taken(min)'
if target_col in df_raw.columns:
    # Parse target
    target = pd.to_numeric(
        df_raw[target_col].astype(str).str.replace(r'[^\d.]', '', regex=True),
        errors='coerce'
    )
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    target.hist(bins=30, ax=axes[0], color='steelblue', edgecolor='black')
    axes[0].set_title('Delivery Time Distribution')
    axes[0].set_xlabel('Minutes')
    
    target.plot(kind='box', ax=axes[1], vert=True)
    axes[1].set_title('Delivery Time Box Plot')
    
    plt.tight_layout()
    plt.show()
    
    print(f'Mean: {target.mean():.1f} min, Median: {target.median():.1f} min')

## 4. Traffic & Weather Impact

In [ ]:
# Clean first to get numeric target
df_clean = clean_delivery(df_raw, config)

if 'traffic_ordinal' in df_clean.columns and 'time_taken_min' in df_clean.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    traffic_map = {0: 'Low', 1: 'Medium', 2: 'High', 3: 'Jam'}
    df_clean['traffic_label'] = df_clean['traffic_ordinal'].map(traffic_map)
    df_clean.boxplot(column='time_taken_min', by='traffic_label', ax=axes[0])
    axes[0].set_title('Delivery Time by Traffic')
    axes[0].set_xlabel('Traffic Density')
    axes[0].set_ylabel('Minutes')
    
    if 'weatherconditions' in df_clean.columns:
        df_clean.boxplot(column='time_taken_min', by='weatherconditions', ax=axes[1])
        axes[1].set_title('Delivery Time by Weather')
        axes[1].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

## 5. Distance Analysis

In [ ]:
from dabba.features.geo import haversine_distance

lat_cols = [c for c in df_clean.columns if 'restaurant_latitude' in c]
lon_cols = [c for c in df_clean.columns if 'restaurant_longitude' in c]
del_lat_cols = [c for c in df_clean.columns if 'delivery_location_latitude' in c]
del_lon_cols = [c for c in df_clean.columns if 'delivery_location_longitude' in c]

if lat_cols and del_lat_cols:
    distances = haversine_distance(
        df_clean[lat_cols[0]].values, df_clean[lon_cols[0]].values,
        df_clean[del_lat_cols[0]].values, df_clean[del_lon_cols[0]].values
    )
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    pd.Series(distances).hist(bins=30, ax=axes[0], color='coral', edgecolor='black')
    axes[0].set_title('Haversine Distance Distribution')
    axes[0].set_xlabel('km')
    
    if 'time_taken_min' in df_clean.columns:
        axes[1].scatter(distances, df_clean['time_taken_min'], alpha=0.1, s=5)
        axes[1].set_xlabel('Distance (km)')
        axes[1].set_ylabel('Delivery Time (min)')
        axes[1].set_title('Distance vs Delivery Time')
    
    plt.tight_layout()
    plt.show()
    
    print(f'Mean distance: {distances.mean():.2f} km')

## 6. Festival & Weekend Effects

In [ ]:
if 'is_festival' in df_clean.columns and 'time_taken_min' in df_clean.columns:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    df_clean.groupby('is_festival')['time_taken_min'].mean().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral'])
    axes[0].set_title('Avg Delivery Time: Festival vs Non-Festival')
    axes[0].set_xticklabels(['No Festival', 'Festival'], rotation=0)
    
    if 'is_weekend' in df_clean.columns:
        df_clean.groupby('is_weekend')['time_taken_min'].mean().plot(kind='bar', ax=axes[1], color=['steelblue', 'coral'])
        axes[1].set_title('Avg Delivery Time: Weekday vs Weekend')
        axes[1].set_xticklabels(['Weekday', 'Weekend'], rotation=0)
    
    plt.tight_layout()
    plt.show()

## Key Takeaways

- Document delivery time patterns here
- Note which features have the strongest relationship with delivery time
- Flag any data quality issues for the modeling stage